In [ ]:
import kagglehub
import os

In [ ]:
# Download the dataset
print("Downloading BraTS 2024 dataset...")
dataset_path = kagglehub.dataset_download("nguyenthanhkhanh/brats2024-small-dataset")
print(f"Dataset downloaded to: {dataset_path}")

In [ ]:
# Inspect the downloaded dataset
print("\nContents of the dataset directory:")
try:
    for item in os.listdir(dataset_path):
        print(item)
    
    # Further inspect a potential subdirectory if it seems relevant (e.g., a common pattern is dataset_path/dataset_name/)
    # This is a guess; actual exploration might reveal a different structure.
    # We'll try to list contents of the first directory found if it's not a file.
    # More sophisticated exploration would be done in subsequent interactive steps if needed.
    first_item_path = os.path.join(dataset_path, os.listdir(dataset_path)[0])
    if os.path.isdir(first_item_path):
        print(f"\nContents of the subdirectory: {first_item_path}")
        for sub_item in os.listdir(first_item_path):
            print(sub_item)

In [ ]:
except FileNotFoundError:
    print(f"Error: Could not find the dataset path: {dataset_path}")
except Exception as e:
    print(f"An error occurred during dataset inspection: {e}")

In [ ]:
# Dependencies and Setup
import argparse
import os
from functools import partial
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.parallel
import torch.utils.data.distributed

# MONAI imports
from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import SwinUNETR
from monai.transforms import Activations, AsDiscrete, Compose
from monai.utils.enums import MetricReduction

# Custom utilities (ensure these paths are correct)
# These .py files should be in SwinUNETR/BRATS24/utils/
from utils.data_utils_brats24 import get_loader, datafold_read 
# Note: datafold_read is imported here if you need to call it directly, 
# but get_loader uses it internally.
# User needs to ensure datafold_read in data_utils_brats24.py works with the new JSON.
# Specifically, the line `json_data = json_data[key]` might need to become
# `json_data = json_data[outer_key][inner_key_like_training]` depending on brats24_folds.json structure.
# The generate_brats24_json.py script creates `{"brats2024_data": {"training": [...]}}`.
# So, datafold_read should be called with basedir, fold, key="brats2024_data", and then inside it access list via json_data['training']
# The get_loader in data_utils_brats24.py calls datafold_read with key=args.json_key (new arg) or hardcoded.
# For now, let's assume get_loader is adapted or args provide enough info.

from utils.trainer_brats24 import run_training 
# We also need LinearWarmupCosineAnnealingLR if that's not in MONAI directly
# Check optimizers/lr_scheduler.py in BRATS21. For now, assume it might be part of MONAI or a simpler scheduler is used.
# The original main.py imports: from optimizers.lr_scheduler import LinearWarmupCosineAnnealingLR
# We should copy that optimizer directory too or simplify.
# For this step, let's try to use a standard PyTorch scheduler if LinearWarmupCosineAnnealingLR is not readily available.

print("Imports successful.")

In [ ]:
# Arguments / Configuration
# Simulate argparse arguments for notebook environment
class Args:
    def __init__(self):
        # Paths and Data
        self.data_dir = "/path/to/your/brats2024-small-dataset"  # USER MUST UPDATE THIS after running earlier download cell
        self.json_list = "./jsons/brats24_folds.json" # Path to the new JSON file
        self.logdir = "./runs_brats24/"
        self.pretrained_dir = "./pretrained_models_brats24/" # Directory for saving/loading checkpoints
        self.fold = 0 # Data fold to use for validation
        self.save_checkpoint = True
        self.resume_ckpt = False # Set to True to resume from a checkpoint
        self.checkpoint = None # Path to checkpoint if resuming, e.g., os.path.join(self.pretrained_dir, "model.pt")
        
        # Model Parameters
        self.feature_size = 48
        self.in_channels = 4 # Typically Flair, T1c, T1, T2
        self.out_channels = 3 # TC, WT, ET
        self.use_checkpoint = False # Gradient checkpointing in SwinUNETR
        self.swin_usr_outside_pretrained_path = "" # if using custom pretrained weights for swin transformer parts

        # Training Parameters
        self.max_epochs = 300 # Adjust as needed
        self.batch_size = 1 # Per GPU
        self.optim_lr = 1e-4
        self.optim_name = "adamw" # adamw, sgd, adam
        self.reg_weight = 1e-5 # Weight decay
        self.momentum = 0.99 # For SGD
        self.lrschedule = "warmup_cosine" # "warmup_cosine" or "cosine_anneal" or None
        self.warmup_epochs = 50
        
        # Validation and Inference
        self.val_every = 10 # Validate every N epochs, original was 100, might be too long for small dataset
        self.sw_batch_size = 4 # Sliding window batch size for validation
        self.roi_x = 96
        self.roi_y = 96
        self.roi_z = 96
        self.infer_overlap = 0.5

        # Augmentation Probabilities (from BRATS21 defaults)
        self.RandFlipd_prob = 0.2
        self.RandRotate90d_prob = 0.2
        self.RandScaleIntensityd_prob = 0.1
        self.RandShiftIntensityd_prob = 0.1
        
        # Data Handling
        self.workers = 8 # Dataloader workers
        self.cache_dataset = False # MONAI CacheDataset, use if I/O is slow
        self.norm_name = "instance" # Normalization type
        self.a_min = -175.0 # ScaleIntensityRanged params
        self.a_max = 250.0
        self.b_min = 0.0
        self.b_max = 1.0
        self.space_x = 1.0 # Original BRATS21 used 1.5, 1.5, 2.0. Check BraTS2024 voxel spacing. For now, assume 1.0 for simplicity.
        self.space_y = 1.0
        self.space_z = 1.0
        self.spatial_dims = 3

        # Loss params
        self.squared_dice = False
        self.smooth_nr = 0.0
        self.smooth_dr = 1e-6

        # Distributed training (simplified for notebook, assuming single GPU or CPU)
        self.distributed = False
        self.world_size = 1
        self.rank = 0
        self.dist_url = "tcp://127.0.0.1:23456"
        self.dist_backend = "nccl"
        self.amp = not True # True to use AMP, original: not args.noamp. Forcing True for now.

        # Test mode (set to True if running inference on validation set)
        self.test_mode = False
        
        # Key for datafold_read, specific to how brats24_folds.json is structured
        # The generate_brats24_json.py creates: {"brats2024_data": {"training": [...]}}
        # So, the outer key is "brats2024_data", and the list is under "training"
        # data_utils_brats24.py's get_loader calls datafold_read(..., key=args.json_key_main)
        # And inside datafold_read, it would access json_data[key_main]['training']
        # This needs careful coordination. Let's add a new arg for this.
        self.json_main_key = "brats2024_data" # The main key in the JSON file that holds the 'training' list.
                                             # This will be used by the (modified) datafold_read in data_utils_brats24.py
                                             # Default in original get_loader was key="training"

args = Args()

# Create directories if they don't exist
os.makedirs(args.logdir, exist_ok=True)
os.makedirs(args.pretrained_dir, exist_ok=True)

print("Arguments configured.")
print(f"USER ACTION: Update 'args.data_dir' with the actual path to the downloaded BraTS 2024 dataset!")
print(f"Dataset JSON: {args.json_list}")
if not os.path.exists(args.json_list):
    print(f"WARNING: {args.json_list} does not exist. Generate it using generate_brats24_json.py after downloading data.")

In [ ]:
# Setup Device
if args.distributed:
    # Simplified distributed setup for notebook, actual main.py has more complex mp.spawn
    if "SLURM_PROCID" in os.environ: # Check if running in a Slurm environment
        args.rank = int(os.environ["SLURM_PROCID"])
        args.gpu = args.rank % torch.cuda.device_count()
    dist.init_process_group(
        backend=args.dist_backend, init_method=args.dist_url, world_size=args.world_size, rank=args.rank
    )
else:
    args.gpu = 0 # Default to GPU 0 if not distributed

if torch.cuda.is_available():
    device = torch.device(f"cuda:{args.gpu}")
    torch.cuda.set_device(device)
    print(f"Using GPU: {device}")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
# Data Loaders
# Ensure data_utils_brats24.py and its datafold_read are compatible with args.json_list and args.json_main_key
# The get_loader function in data_utils_brats24.py might need to be modified to use args.json_main_key
# when it calls datafold_read. For example:
# train_files, validation_files = datafold_read(datalist=datalist_json, basedir=data_dir, fold=args.fold, key=args.json_main_key)
# And then inside datafold_read:
#   with open(datalist) as f: json_content = json.load(f)
#   actual_list = json_content[key]['training'] # Access the list

print("Loading data...")
loader = get_loader(args) # loader[0] is train, loader[1] is val (if not test_mode)
print("Data loaders ready.")

In [ ]:
# Model Initialization
print("Initializing model...")
model = SwinUNETR(
    in_channels=args.in_channels,
    out_channels=args.out_channels,
    img_size=(args.roi_x, args.roi_y, args.roi_z), # img_size is expected by SwinUNETR
    feature_size=args.feature_size,
    use_checkpoint=args.use_checkpoint,
    # spatial_dims=args.spatial_dims # Already 3 by default
).to(device)

if args.swin_usr_outside_pretrained_path and os.path.exists(args.swin_usr_outside_pretrained_path):
    print(f"Loading Swin Transformer custom weights from: {args.swin_usr_outside_pretrained_path}")
    model.load_swin_usr_outside_weights(args.swin_usr_outside_pretrained_path)
elif args.resume_ckpt and args.checkpoint and os.path.exists(args.checkpoint):
    print(f"Resuming training from checkpoint: {args.checkpoint}")
    checkpoint_data = torch.load(args.checkpoint, map_location=device)
    # Handle potential DistributedDataParallel prefix
    state_dict = checkpoint_data.get("state_dict", checkpoint_data)
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            new_state_dict[k[7:]] = v # remove `module.`
        else:
            new_state_dict[k] = v
    model.load_state_dict(new_state_dict)
else:
    print("Initializing model with random weights (or default SwinUNETR pretraining if applicable).")

pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {pytorch_total_params / 1e6:.2f}M")

In [ ]:
# Loss, Optimizer, Scheduler
print("Setting up loss, optimizer, and scheduler...")
if args.squared_dice:
    dice_loss = DiceLoss(
        to_onehot_y=False, sigmoid=True, squared_pred=True, 
        smooth_nr=args.smooth_nr, smooth_dr=args.smooth_dr
    )
else:
    dice_loss = DiceLoss(to_onehot_y=False, sigmoid=True, smooth_nr=args.smooth_nr, smooth_dr=args.smooth_dr)

post_sigmoid = Activations(sigmoid=True)
post_pred = AsDiscrete(argmax=False, logit_thresh=0.5) # logit_thresh for BRATS like tasks often 0.5

dice_acc = DiceMetric(include_background=True, reduction=MetricReduction.MEAN_BATCH, get_not_nans=True)

model_inferer = partial(
    sliding_window_inference,
    roi_size=(args.roi_x, args.roi_y, args.roi_z),
    sw_batch_size=args.sw_batch_size,
    predictor=model,
    overlap=args.infer_overlap,
    device=device, # Specify device for sliding_window_inference
    progress=True   # Show progress bar
)

# Optimizer
if args.optim_name == "adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=args.optim_lr, weight_decay=args.reg_weight)
elif args.optim_name == "adamw":
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.optim_lr, weight_decay=args.reg_weight)
elif args.optim_name == "sgd":
    optimizer = torch.optim.SGD(
        model.parameters(), lr=args.optim_lr, momentum=args.momentum, nesterov=True, weight_decay=args.reg_weight
    )
else:
    raise ValueError(f"Unsupported Optimization Algorithm: {args.optim_name}")

# Scheduler
if args.lrschedule == "warmup_cosine":
    # The original project had a custom LinearWarmupCosineAnnealingLR.
    # MONAI >= 0.8.0 has WarmupCosineSchedule. For simplicity, let's use CosineAnnealingLR.
    # If LinearWarmupCosineAnnealingLR is critical, its code needs to be copied to utils.
    # For now, using standard PyTorch scheduler.
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.max_epochs - args.warmup_epochs if args.warmup_epochs > 0 else args.max_epochs)
    # A proper warmup_cosine usually involves composing schedulers or using a more specific one.
    # Let's try a simpler CosineAnnealingLR first. User can refine this.
    # If using MONAI's: from monai.optimizers import WarmupCosineSchedule
    # scheduler = WarmupCosineSchedule(optimizer, warmup_steps=args.warmup_epochs, t_total=args.max_epochs)
    # For now, let's use a basic one from PyTorch to avoid adding more file dependencies yet.
    print("Using CosineAnnealingLR. For LinearWarmupCosineAnnealingLR, ensure 'optimizers/lr_scheduler.py' is copied and imported.")
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.max_epochs)

elif args.lrschedule == "cosine_anneal":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.max_epochs)
else:
    scheduler = None

start_epoch = 0
if args.resume_ckpt and args.checkpoint and os.path.exists(args.checkpoint) and "epoch" in checkpoint_data:
    start_epoch = checkpoint_data["epoch"] + 1
    if scheduler and "scheduler" in checkpoint_data: # Load scheduler state
        scheduler.load_state_dict(checkpoint_data["scheduler"])
    print(f"Resuming from epoch {start_epoch}")


print("Setup complete.")

In [ ]:
# Training Run
print("Starting training process...")
# Semantic classes for logging during validation, matching BRATS typical outputs
# WT (Whole Tumor), TC (Tumor Core), ET (Enhancing Tumor)
# Order might depend on ConvertToMultiChannelBasedOnBratsClassesd output
semantic_classes = ["Dice_Val_WT", "Dice_Val_TC", "Dice_Val_ET"] # Verify order based on label conversion

# Ensure model is on the correct device for distributed training if ever enabled
if args.distributed:
    model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu], output_device=args.gpu)


# Check if loader is a list (train_loader, val_loader) or single (test_loader)
train_loader_to_use = loader[0] if isinstance(loader, list) and not args.test_mode else None
val_loader_to_use = loader[1] if isinstance(loader, list) and not args.test_mode else loader

if args.test_mode:
    print("Running in Test Mode: Only validation/testing will be performed if a checkpoint is loaded.")
    # (Logic for test mode / inference would go here - usually involves loading a checkpoint and running val_epoch)
    # For now, this notebook focuses on setting up training.
    if args.checkpoint and os.path.exists(args.checkpoint):
        # Perform a validation run
        val_acc = val_epoch( # This function is in trainer_brats24.py, need to ensure it's imported/available
            model,
            val_loader_to_use,
            epoch=0, # Or current epoch from checkpoint
            acc_func=dice_acc,
            args=args,
            model_inferer=model_inferer,
            post_sigmoid=post_sigmoid,
            post_pred=post_pred,
        )
        print(f"Test mode validation accuracy: {val_acc}")
    else:
        print("Test mode selected, but no checkpoint provided or found. Exiting.")

elif train_loader_to_use and val_loader_to_use:
    print(f"Starting training from epoch {start_epoch} for {args.max_epochs} epochs.")
    accuracy = run_training(
        model=model,
        train_loader=train_loader_to_use,
        val_loader=val_loader_to_use,
        optimizer=optimizer,
        loss_func=dice_loss,
        acc_func=dice_acc,
        args=args,
        model_inferer=model_inferer,
        scheduler=scheduler,
        start_epoch=start_epoch,
        post_sigmoid=post_sigmoid,
        post_pred=post_pred,
        semantic_classes=semantic_classes, # Pass to run_training
        device=device # Pass device to run_training
    )
    print(f"Training finished. Final best accuracy: {accuracy}")
else:
    print("Could not obtain train/validation loaders. Please check data configuration.")

In [ ]:
# Placeholder for potential further actions:
# - Saving final model
# - Running inference on a test set
# - Visualizing results

print("Notebook execution cell for training logic is complete.")
print("Remember to fill in `args.data_dir` with the correct path to your downloaded dataset.")